# Lab 3.11 — Pretrained Models and Transfer Learning
CPU-first, deterministic notebook using scikit-learn's bundled handwritten-digits dataset. First compare frozen transfer with scratch training, then inspect architecture and preprocessing as part of the pretrained-model contract.


In [ ]:
import copy
import random
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

SEED = 13
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

digits = load_digits()
X = digits.data.astype("float32") / 16.0
y = digits.target.astype("int64")
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)

class Encoder(torch.nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.net = torch.nn.Sequential(torch.nn.Linear(64, hidden), torch.nn.ReLU())
    def forward(self, x):
        return self.net(x)

class SourceModel(torch.nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.encoder = Encoder(hidden)
        self.head = torch.nn.Linear(hidden, 10)
    def forward(self, x):
        return self.head(self.encoder(x))

class TargetModel(torch.nn.Module):
    def __init__(self, encoder=None, hidden=32):
        super().__init__()
        self.encoder = encoder if encoder is not None else Encoder(hidden)
        self.head = torch.nn.Linear(hidden, 2)
    def forward(self, x):
        return self.head(self.encoder(x))

def train_classifier(model, X_t, y_t, epochs, lr):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr)
    loss_fn = torch.nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(X_t), y_t)
        loss.backward()
        opt.step()
    return model

def accuracy(model, X_t, y_t):
    model.eval()
    with torch.no_grad():
        return float((model(X_t).argmax(1) == y_t).float().mean())

X_source = torch.tensor(X_train)
y_source = torch.tensor(y_train)
X_valid = torch.tensor(X_val)
y_valid_source = torch.tensor(y_val)
torch.manual_seed(SEED)
source = train_classifier(SourceModel(), X_source, y_source, epochs=120, lr=0.1)
source_accuracy = accuracy(source, X_valid, y_valid_source)
loop_digits = np.array([0, 6, 8, 9])
y_target_all = np.isin(y_train, loop_digits).astype("int64")
y_target_valid = np.isin(y_val, loop_digits).astype("int64")
all_idx = np.arange(len(y_train))
small_idx, _ = train_test_split(all_idx, train_size=40, random_state=SEED, stratify=y_target_all)
X_target = torch.tensor(X_train[small_idx])
y_target = torch.tensor(y_target_all[small_idx])
y_valid_target = torch.tensor(y_target_valid)

def make_frozen_transfer():
    encoder = copy.deepcopy(source.encoder)
    for p in encoder.parameters():
        p.requires_grad = False
    torch.manual_seed(SEED)
    return train_classifier(TargetModel(encoder=encoder), X_target, y_target, epochs=200, lr=0.2)

def make_scratch():
    torch.manual_seed(SEED)
    return train_classifier(TargetModel(), X_target, y_target, epochs=200, lr=0.2)


In [ ]:
frozen = make_frozen_transfer()
scratch = make_scratch()
frozen_acc = accuracy(frozen, X_valid, y_valid_target)
scratch_acc = accuracy(scratch, X_valid, y_valid_target)
print(f"source_accuracy={source_accuracy:.3f}")
print(f"frozen_transfer_accuracy={frozen_acc:.3f}")
print(f"scratch_accuracy={scratch_acc:.3f}")
assert source_accuracy > 0.80
assert frozen_acc > 0.75


## Part A — Change target data size
Reduce `train_size=40` to `20`. Predict first, then rerun with the same validation set and seed. Restore `40` afterward.


In [ ]:
saved = copy.deepcopy(source.encoder.state_dict())
good = Encoder(32)
good.load_state_dict(saved)
good.eval()
wrong_preprocessing_offset = 2.0
with torch.no_grad():
    correct_features = good(X_valid)
    wrong_features = good(X_valid + wrong_preprocessing_offset)
    shift = float((correct_features - wrong_features).abs().mean())
shape_mismatch_detected = False
try:
    bad = Encoder(24)
    bad.load_state_dict(saved)
except RuntimeError:
    shape_mismatch_detected = True
print(f"mean_feature_shift_from_wrong_preprocessing={shift:.3f}")
print(f"shape_mismatch_detected={shape_mismatch_detected}")
assert shape_mismatch_detected
assert shift > 0.1


## Part B — Inspect the pretrained contract
Change `wrong_preprocessing_offset = 2.0` to `0.5`. Predict whether the feature shift should shrink, rerun, and restore `2.0`. Record architecture, weight identity, preprocessing, label mapping, provenance, evaluation mode, and known limitations beside the checkpoint.
